# Gaussian Shading 官方参考环境复现与受治理导入入口

该 Notebook 服务第二条链路: 官方参考环境复现 + 受治理导入。它不把基于 Stable Diffusion 2.1 的参考结果混入 SD3.5 主表, 只生成补充表所需的官方参考记录、环境报告、命令日志、schema、validation report 和 Google Drive 压缩包。

运行依赖: 不依赖主方法前序结果包, 但必须共享当前 `SLM_WM_PAPER_RUN_NAME` 派生的 prompt split、样本数和目标 FPR。四个 official reference 入口之间互不依赖, 可以在独立 Colab 会话中并行运行; 完整结果包闭合时会从 `external_baseline_official_reference` Drive 子目录读取这些包。

Notebook 通过统一宿主 launcher 的精确父解释器分派到 `paper_experiments/runners/gaussian_shading_official_reference.py`, 只负责 Colab 冷启动、参数注入、远程执行和打包。

## 运行边界

1. `probe_paper`、`pilot_paper` 与 `full_paper` 分别使用 FPR=0.1、FPR=0.01和 FPR=0.001, Prompt 数量分别为70、700和7000; 三个层级执行同一完整官方参考协议与统计实现。
2. helper 按 `external_baseline/source_registry.json` 核验 Gaussian Shading 官方源码的精确 Git commit、远端地址和确定性补丁工作树。
3. 扩散模型固定为 `Manojb/stable-diffusion-2-1-base@0094d483a120f3f33dafbd187ea4aa60d10de75c`, 并在 `/content/slm_wm_model_sources` 共享快照中执行逐文件摘要核验。
4. Prompt 数据固定为 `Gustavosta/Stable-Diffusion-Prompts@d816d4a05cb89bde39dd99284c459801e1e7e69a`; 数据 revision 会进入源码补丁证据、正式记录和运行摘要。
5. OpenCLIP 固定使用 `ViT-g-14` 的本地 `open_clip_pytorch_model.bin`; runner 会核验来源 revision、文件大小、SHA-256 和共享快照摘要后再启动官方命令。
6. 官方脚本在固定依赖配置的隔离 Python 3.8 环境中运行, Notebook 只读取受治理结果, 精确父解释器负责 repository 调度、指标解析与证据生成。
7. 当前会话必须真实执行官方命令并返回0, 且检测、追踪、bit accuracy 与 CLIP 科学指标必须完整; 任一条件失败都会写出诊断并阻断正式记录与归档。

运行期间, repository 共享持久化会话会把稳定文件周期写入当前 workflow 的 Drive 目录, 并在恢复前复验 formal execution commit、科学 profile、配置身份与逐文件 SHA-256. checkpoint 仅用于续跑, 不属于正式论文证据。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("正式运行前必须通过 SLM_WM_REPOSITORY_COMMIT 提供精确40位小写 Git SHA")
workspace_dir = Path(os.environ.get("SLM_WM_WORKSPACE_DIR", "/content/slm_wm_repository"))
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)

status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)

os.chdir(workspace_dir)
print({"workspace_dir": str(workspace_dir), "repository_commit": repository_commit})


In [ ]:
FORMAL_HOST_LAUNCHER = "scripts/run_formal_workflow_host.py"
print({"formal_host_launcher": FORMAL_HOST_LAUNCHER})


In [ ]:
# 论文运行配置由精确 workflow_orchestrator 子解释器统一加载.


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get("HF_TOKEN")
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = token_from_secret
print({"hf_token_available_for_scientific_child": bool(os.environ.get("HF_TOKEN"))})


In [ ]:
import shutil
import subprocess

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi is None:
    raise RuntimeError('当前 Colab runtime 缺少 nvidia-smi, 无法执行 CUDA 科学工作流')
device_query = subprocess.run(
    [nvidia_smi, '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    check=False,
    capture_output=True,
    text=True,
)
if device_query.returncode != 0:
    raise RuntimeError('nvidia-smi 查询失败: ' + device_query.stderr.strip())
print(device_query.stdout.strip())


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

workflow_route = 'official_reference_gaussian_shading'
result_path = Path("outputs/formal_workflow_execution") / SLM_WM_PAPER_RUN_NAME / workflow_route / "workflow_result.json"
host_command = [
    sys.executable,
    "-I",
    FORMAL_HOST_LAUNCHER,
    "--root",
    ".",
    "--repository-commit",
    repository_commit,
    "gpu",
    "--workflow",
    workflow_route,
    "--paper-run-name",
    SLM_WM_PAPER_RUN_NAME,
    "--result-path",
    result_path.as_posix(),
]
subprocess.run(host_command, check=True)
formal_workflow_result = json.loads(result_path.read_text(encoding="utf-8"))
assert formal_workflow_result["decision"] == "pass", formal_workflow_result
formal_workflow_result


In [ ]:
print(json.dumps(formal_workflow_result, ensure_ascii=False, sort_keys=True, indent=2))
